# Statistical Evaluation of LLM Calibration Methods

This notebook demonstrates the evaluation pipeline for comparing three calibration methods for LLM classifiers:
- **Uncalibrated baseline**: Raw softmax probabilities from the model
- **Temperature scaling**: Standard calibration method (Guo et al. 2017)
- **Thermodynamic entropy calibration**: Proposed method using per-sample temperature based on entropy

## What this notebook does:
1. Loads pre-computed predictions from an SST-2 sentiment classification experiment
2. Computes calibration metrics (ECE, Brier Score, NLL, Accuracy) with bootstrap confidence intervals
3. Performs statistical significance tests (Wilcoxon, Bootstrap, Cohen's d)
4. Generates reliability diagrams for visual calibration assessment
5. Analyzes ECE decomposition by confidence bins
6. Plots accuracy-calibration tradeoff curves

## Metrics explained:
- **ECE (Expected Calibration Error)**: Lower is better. Measures difference between confidence and accuracy.
- **Brier Score**: Lower is better. Penalizes both miscalibration and poor accuracy.
- **NLL (Negative Log-Likelihood)**: Lower is better. Measures pure probabilistic quality.
- **Accuracy**: Higher is better. Percentage of correct predictions.

## Dataset:
- SST-2 (Stanford Sentiment Treebank): Binary sentiment classification (positive/negative)
- Using a mini subset (5 examples) for fast demo execution

In [ ]:
# Install dependencies - follows aii-colab pattern for Colab compatibility
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

print('Dependencies installed successfully!')

In [ ]:
# Imports - copied from original eval.py with minimal additions for notebook
from loguru import logger
from pathlib import Path
import json
import sys
import numpy as np
from scipy.stats import wilcoxon, norm
from sklearn.metrics import brier_score_loss, log_loss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# NumPy 2.0 compatibility shim (in case non-Colab packages need it)
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

# Configure logger for notebook (output to stdout only)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

print('All imports loaded successfully!')

In [ ]:
# Data loading helper - uses GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d144fc-thermodynamic-entropy-calibration-for-im/main/round-2/evaluation-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load mini demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL failed: {e}")
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading helper defined!')

In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")
print(f"Dataset: {data['datasets'][0]['dataset']}")
print(f"Methods: {data['metadata']['methods']}")

## Configuration

All tunable parameters are defined here. For the demo, we use MINIMAL values to ensure fast execution.

- `n_bootstrap`: Number of bootstrap iterations for confidence intervals (1000 original → 10 for demo)
- `n_bins`: Number of bins for ECE computation (keep at 10)
- `n_examples`: Number of examples to use (175 original → 5 for demo)

In [ ]:
# Configuration - MINIMAL values for fast demo execution
# To run with more realistic parameters, increase these values

# Bootstrap iterations (original: 1000, demo: 10)
n_bootstrap = 10  # @param {type:"slider", min:10, max:1000, step:10}

# Number of bins for ECE calculation
n_bins = 10

# Number of examples to use (original: 175, demo: 5)
# Set to None to use all examples in the loaded data
n_examples = 5  # @param {type:"slider", min:2, max:175, step:1}

print(f"Config: n_bootstrap={n_bootstrap}, n_bins={n_bins}, n_examples={n_examples}")

## Utility Functions

These functions are copied directly from the original `eval.py` script. They compute calibration metrics, perform bootstrap analysis, and generate reliability diagrams.

In [ ]:
# Utility Functions - copied exactly from eval.py

def parse_prob_string(prob_str: str) -> np.ndarray:
    """Parse probability string to numpy array."""
    return np.array(json.loads(prob_str))

def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    """
    Compute Expected Calibration Error (ECE).
    """
    pred_confs = np.max(probs, axis=1)
    pred_classes = np.argmax(probs, axis=1)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        mask = (pred_confs >= bins[i]) & (pred_confs < bins[i + 1])
        if i == n_bins - 1:
            mask = (pred_confs >= bins[i]) & (pred_confs <= bins[i + 1])

        if np.sum(mask) > 0:
            bin_acc = np.mean(pred_classes[mask] == labels[mask])
            bin_conf = np.mean(pred_confs[mask])
            weight = np.sum(mask) / len(labels)
            ece += weight * abs(bin_acc - bin_conf)

    return ece

def compute_ece_decomposition(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> Dict:
    """Compute ECE decomposition by confidence bins."""
    pred_confs = np.max(probs, axis=1)
    pred_classes = np.argmax(probs, axis=1)

    bins = np.linspace(0, 1, n_bins + 1)
    decomposition = []

    for i in range(n_bins):
        mask = (pred_confs >= bins[i]) & (pred_confs < bins[i + 1])
        if i == n_bins - 1:
            mask = (pred_confs >= bins[i]) & (pred_confs <= bins[i + 1])

        if np.sum(mask) > 0:
            bin_acc = np.mean(pred_classes[mask] == labels[mask])
            bin_conf = np.mean(pred_confs[mask])
            bin_count = np.sum(mask)
            bin_ece = abs(bin_acc - bin_conf)

            decomposition.append({
                "bin_index": i,
                "bin_range": [float(bins[i]), float(bins[i + 1])],
                "count": int(bin_count),
                "accuracy": float(bin_acc),
                "confidence": float(bin_conf),
                "ece_contribution": float(bin_ece * bin_count / len(labels))
            })
        else:
            decomposition.append({
                "bin_index": i,
                "bin_range": [float(bins[i]), float(bins[i + 1])],
                "count": 0,
                "accuracy": 0.0,
                "confidence": 0.0,
                "ece_contribution": 0.0
            })

    return {"bins": decomposition}

def compute_brier_score(probs: np.ndarray, labels: np.ndarray) -> float:
    """Compute Brier Score for multi-class."""
    n_samples, n_classes = probs.shape
    y_true = np.zeros((n_samples, n_classes))
    y_true[np.arange(n_samples), labels] = 1
    return np.mean(np.sum((probs - y_true) ** 2, axis=1))

def compute_nll(probs: np.ndarray, labels: np.ndarray) -> float:
    """Compute Negative Log-Likelihood."""
    true_class_probs = probs[np.arange(len(labels)), labels]
    true_class_probs = np.clip(true_class_probs, 1e-15, 1.0)
    return -np.mean(np.log(true_class_probs))

def compute_accuracy(probs: np.ndarray, labels: np.ndarray) -> float:
    """Compute accuracy."""
    pred_classes = np.argmax(probs, axis=1)
    return np.mean(pred_classes == labels)

def bootstrap_metric(metric_func, probs: np.ndarray, labels: np.ndarray,
                     n_bootstrap: int = 1000, confidence: float = 0.95,
                     **kwargs) -> Tuple[float, float, float]:
    """Compute metric with bootstrap confidence interval."""
    n_samples = len(labels)
    point_estimate = metric_func(probs, labels, **kwargs)

    bootstrap_values = []
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        bootstrap_probs = probs[indices]
        bootstrap_labels = labels[indices]
        bootstrap_values.append(metric_func(bootstrap_probs, bootstrap_labels, **kwargs))

    bootstrap_values = np.array(bootstrap_values)

    alpha = (1 - confidence) / 2
    lower = np.percentile(bootstrap_values, 100 * alpha)
    upper = np.percentile(bootstrap_values, 100 * (1 - alpha))

    return point_estimate, lower, upper

def bootstrap_paired_difference(probs1: np.ndarray, probs2: np.ndarray,
                                labels: np.ndarray, metric_func,
                                n_bootstrap: int = 1000) -> Tuple[float, float, float]:
    """Bootstrap hypothesis test for paired difference in metrics."""
    n_samples = len(labels)

    obs_diff = metric_func(probs1, labels) - metric_func(probs2, labels)

    bootstrap_diffs = []
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        diff = metric_func(probs1[indices], labels[indices]) - metric_func(probs2[indices], labels[indices])
        bootstrap_diffs.append(diff)

    bootstrap_diffs = np.array(bootstrap_diffs)

    p_value = np.mean(np.abs(bootstrap_diffs) >= np.abs(obs_diff))

    lower = np.percentile(bootstrap_diffs, 2.5)
    upper = np.percentile(bootstrap_diffs, 97.5)

    return obs_diff, p_value, lower, upper

def cohens_d(x1: np.ndarray, x2: np.ndarray) -> float:
    """Compute Cohen's d for paired samples."""
    diff = x1 - x2
    return np.mean(diff) / np.std(diff, ddof=1)

print('Utility functions defined!')

## Step 1: Load and Parse Data

Extract predictions and labels from the loaded data. The data contains pre-computed probabilities for each calibration method.

In [ ]:
# Step 1: Load and parse data from the demo data
# This replaces reading from 'experiment_out.json' in the original script

exp_data = data  # Use the loaded demo data

# Extract dataset info
dataset_name = exp_data["datasets"][0]["dataset"]
examples = exp_data["datasets"][0]["examples"]

# Limit number of examples if specified
if n_examples is not None:
    examples = examples[:n_examples]

logger.info(f"Processing {len(examples)} examples from {dataset_name}")

# Parse predictions
methods = ["uncalibrated", "temperature_scaling", "thermodynamic_entropy"]
method_probs = {}
method_prob_keys = {
    "uncalibrated": "metadata_uncalibrated_probs",
    "temperature_scaling": "metadata_ts_probs",
    "thermodynamic_entropy": "metadata_te_probs"
}
labels = []

for method in methods:
    method_probs[method] = []

for i, example in enumerate(examples):
    # Parse true label
    true_label_str = example["output"].split(": ")[1].strip()
    true_label = int(true_label_str)
    labels.append(true_label)

    # Parse probabilities for each method
    for method in methods:
        prob_key = method_prob_keys[method]
        if prob_key in example:
            probs = parse_prob_string(example[prob_key])
            method_probs[method].append(probs)

labels = np.array(labels)
for method in methods:
    method_probs[method] = np.array(method_probs[method])

logger.info(f"Loaded predictions for {len(methods)} methods")
logger.info(f"Labels shape: {labels.shape}")
for method in methods:
    logger.info(f"{method} probs shape: {method_probs[method].shape}")

## Step 2: Compute Metrics with Bootstrap CI

Calculate calibration metrics with 95% bootstrap confidence intervals for each method.

In [ ]:
# Step 2: Compute metrics with bootstrap confidence intervals

results = {}

for method in methods:
    logger.info(f"\nEvaluating {method}...")

    probs = method_probs[method]

    # ECE
    ece, ece_lower, ece_upper = bootstrap_metric(
        compute_ece, probs, labels, n_bootstrap=n_bootstrap, n_bins=n_bins
    )

    # Brier Score
    brier, brier_lower, brier_upper = bootstrap_metric(
        compute_brier_score, probs, labels, n_bootstrap=n_bootstrap
    )

    # NLL
    nll, nll_lower, nll_upper = bootstrap_metric(
        compute_nll, probs, labels, n_bootstrap=n_bootstrap
    )

    # Accuracy
    acc, acc_lower, acc_upper = bootstrap_metric(
        compute_accuracy, probs, labels, n_bootstrap=n_bootstrap
    )

    results[method] = {
        "ece": {"value": ece, "ci_lower": ece_lower, "ci_upper": ece_upper},
        "brier": {"value": brier, "ci_lower": brier_lower, "ci_upper": brier_upper},
        "nll": {"value": nll, "ci_lower": nll_lower, "ci_upper": nll_upper},
        "accuracy": {"value": acc, "ci_lower": acc_lower, "ci_upper": acc_upper}
    }

    logger.info(f"  ECE: {ece:.4f} [{ece_lower:.4f}, {ece_upper:.4f}]")
    logger.info(f"  Brier: {brier:.4f} [{brier_lower:.4f}, {brier_upper:.4f}]")
    logger.info(f"  NLL: {nll:.4f} [{nll_lower:.4f}, {nll_upper:.4f}]")
    logger.info(f"  Accuracy: {acc:.4f} [{acc_lower:.4f}, {acc_upper:.4f}]")

## Step 3: Statistical Significance Tests

Compare methods using:
- Paired Wilcoxon Signed-Rank Test (per-sample NLL)
- Bootstrap Hypothesis Test (ECE difference)
- Cohen's d (Effect size)

In [ ]:
# Step 3: Statistical significance tests

# Compute per-sample metrics for Wilcoxon test
for method in methods:
    probs = method_probs[method]
    true_class_probs = probs[np.arange(len(labels)), labels]
    true_class_probs = np.clip(true_class_probs, 1e-15, 1.0)
    results[method]["per_sample_nll"] = -np.log(true_class_probs)

    pred_classes = np.argmax(probs, axis=1)
    results[method]["per_sample_acc"] = (pred_classes == labels).astype(float)

# Compare methods using Wilcoxon test
comparisons = [
    ("temperature_scaling", "uncalibrated"),
    ("thermodynamic_entropy", "uncalibrated"),
    ("thermodynamic_entropy", "temperature_scaling")
]

statistical_tests = {}

for method1, method2 in comparisons:
    logger.info(f"\nComparing {method1} vs {method2}...")

    # Wilcoxon test on per-sample NLL
    stat, p_value_nll = wilcoxon(
        results[method1]["per_sample_nll"],
        results[method2]["per_sample_nll"],
        alternative='two-sided'
    )

    # Bootstrap hypothesis test on ECE difference
    ece_diff, p_value_ece, ece_diff_lower, ece_diff_upper = bootstrap_paired_difference(
        method_probs[method1],
        method_probs[method2],
        labels,
        compute_ece,
        n_bootstrap=n_bootstrap
    )

    # Effect size (Cohen's d) for NLL difference
    effect_size = cohens_d(results[method1]["per_sample_nll"], results[method2]["per_sample_nll"])

    statistical_tests[f"{method1}_vs_{method2}"] = {
        "wilcoxon_p_value_nll": float(p_value_nll),
        "bootstrap_p_value_ece": float(p_value_ece),
        "ece_difference": float(ece_diff),
        "ece_diff_ci_lower": float(ece_diff_lower),
        "ece_diff_ci_upper": float(ece_diff_upper),
        "cohens_d_nll": float(effect_size)
    }

    logger.info(f"  Wilcoxon p-value (NLL): {p_value_nll:.4f}")
    logger.info(f"  Bootstrap p-value (ECE): {p_value_ece:.4f}")
    logger.info(f"  ECE difference: {ece_diff:.4f} [{ece_diff_lower:.4f}, {ece_diff_upper:.4f}]")
    logger.info(f"  Cohen's d (NLL): {effect_size:.4f}")

## Step 4: Generate Reliability Diagram Data

Reliability diagrams show calibration visually by plotting accuracy vs confidence in bins.

In [ ]:
# Step 4: Generate reliability diagram data

reliability_data = {}

for method in methods:
    probs = method_probs[method]
    pred_confs = np.max(probs, axis=1)
    pred_classes = np.argmax(probs, axis=1)

    bins = np.linspace(0, 1, n_bins + 1)
    bin_data = []

    for i in range(n_bins):
        mask = (pred_confs >= bins[i]) & (pred_confs < bins[i + 1])
        if i == n_bins - 1:
            mask = (pred_confs >= bins[i]) & (pred_confs <= bins[i + 1])

        if np.sum(mask) > 0:
            bin_acc = np.mean(pred_classes[mask] == labels[mask])
            bin_conf = np.mean(pred_confs[mask])
            bin_count = np.sum(mask)
        else:
            bin_acc = 0.0
            bin_conf = 0.0
            bin_count = 0

        bin_data.append({
            "bin_index": i,
            "bin_range": [float(bins[i]), float(bins[i + 1])],
            "count": int(bin_count),
            "accuracy": float(bin_acc),
            "confidence": float(bin_conf)
        })

    reliability_data[method] = bin_data
    logger.info(f"Reliability data generated for {method}")

print('Reliability diagram data generated!')

## Step 5: ECE Decomposition Analysis

Break down ECE by confidence bins to identify where each method succeeds or fails.

In [ ]:
# Step 5: ECE Decomposition Analysis

ece_decomposition = {}

for method in methods:
    probs = method_probs[method]
    decomp = compute_ece_decomposition(probs, labels, n_bins=n_bins)
    ece_decomposition[method] = decomp

    # Log top contributing bins
    bins_sorted = sorted(decomp["bins"], key=lambda x: x["ece_contribution"], reverse=True)
    logger.info(f"\n{method} - Top ECE contributors:")
    for bin_info in bins_sorted[:3]:
        if bin_info["count"] > 0:
            logger.info(f"  Bin {bin_info['bin_index']} ({bin_info['bin_range'][0]:.1f}-{bin_info['bin_range'][1]:.1f}): "
                      f"count={bin_info['count']}, acc={bin_info['accuracy']:.3f}, "
                      f"conf={bin_info['confidence']:.3f}, ece_contrib={bin_info['ece_contribution']:.4f}")

print('ECE decomposition complete!')

## Results Visualization

Display key results in readable tables and plots.

In [ ]:
# Results Visualization

import matplotlib.pyplot as plt

# 1. Print results table
print("="*78)
print("RESULTS SUMMARY")
print("="*78)
print(f"{'Method':<30} {'ECE':>12} {'Brier':>12} {'NLL':>12} {'Acc':>12}")
print("-" * 78)

for method in methods:
    ece_val = results[method]["ece"]["value"]
    brier_val = results[method]["brier"]["value"]
    nll_val = results[method]["nll"]["value"]
    acc_val = results[method]["accuracy"]["value"]

    print(f"{method:<30} {ece_val:>8.4f} {brier_val:>12.4f} {nll_val:>12.4f} {acc_val:>12.4f}")

print("\n" + "="*78)
print("STATISTICAL SIGNIFICANCE (p-values)")
print("="*78)
for comp, tests in statistical_tests.items():
    print(f"  {comp}: Wilcoxon={tests['wilcoxon_p_value_nll']:.4f}, "
          f"Bootstrap={tests['bootstrap_p_value_ece']:.4f}")

# 2. Plot reliability diagrams
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, method in enumerate(methods):
    ax = axes[idx]

    # Diagonal reference line
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')

    # Reliability curve
    bin_confs = [b['confidence'] for b in reliability_data[method] if b['count'] > 0]
    bin_accs = [b['accuracy'] for b in reliability_data[method] if b['count'] > 0]

    if bin_confs:
        ax.plot(bin_confs, bin_accs, 'o-', linewidth=2, label=method.replace('_', ' ').title())

    ax.set_xlabel('Confidence', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Reliability Diagram: {method.replace("_", " ").title()}', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

# 3. Plot ECE decomposition
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, method in enumerate(methods):
    ax = axes[idx]

    bins_data = ece_decomposition[method]['bins']
    bin_indices = [b['bin_index'] for b in bins_data]
    ece_contribs = [b['ece_contribution'] for b in bins_data]
    counts = [b['count'] for b in bins_data]

    # Bar plot of ECE contributions
    bars = ax.bar(bin_indices, ece_contribs, alpha=0.7, label='ECE Contribution')

    # Color bars by count
    for bar, count in zip(bars, counts):
        if count == 0:
            bar.set_color('lightgray')

    ax.set_xlabel('Bin Index', fontsize=12)
    ax.set_ylabel('ECE Contribution', fontsize=12)
    ax.set_title(f'ECE Decomposition: {method.replace("_", " ").title()}', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_xticks(bin_indices)

plt.tight_layout()
plt.show()

print("\nVisualization complete!")

## Summary

This demo notebook has demonstrated the evaluation pipeline for comparing calibration methods:

1. **Metrics computed**: ECE, Brier Score, NLL, and Accuracy with bootstrap confidence intervals
2. **Statistical tests**: Wilcoxon signed-rank test, bootstrap hypothesis test, and Cohen's d
3. **Visualizations**: Reliability diagrams and ECE decomposition plots

### Key observations from this demo:
- Temperature scaling typically achieves the best calibration (lowest ECE)
- Thermodynamic entropy improves calibration over uncalibrated baseline
- All methods maintain similar accuracy (calibration doesn't affect predictions)
- With only 5 examples, confidence intervals are very wide (use more examples for meaningful results)

### To run with more realistic parameters:
1. Increase `n_bootstrap` to 1000 in the config cell
2. Increase `n_examples` to 175 (or use the full dataset)
3. Re-run all cells

**Note**: The full evaluation with 175 samples and 1000 bootstrap iterations takes ~2-3 minutes to complete.